In [ ]:
from eo_tides.utils import list_models

available_models, supported_models = list_models(directory="tide_models")
print("Available:", available_models)
print("Supported:", supported_models)

In [ ]:
import wxee
import numpy as np
import geopandas as gpd
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import re 
from odc.geo.geobox import GeoBox
from eo_tides.eo import tag_tides
from eo_tides.model import model_tides
import rasterio
import shutil

In [ ]:
def safe_folder_to_df(site_folder, save_csv=True):
    site = Path(site_folder)
    rx = re.compile(r"^S2[AB]_MSIL2A_(\d{8}T\d{6})_")
    rows = []
    for p in sorted(site.iterdir()):
        if p.is_dir() and p.name.endswith(".SAFE"):
            m = rx.match(p.name)
            if m:
                t = pd.to_datetime(m.group(1), format="%Y%m%dT%H%M%S", utc=True)
                rows.append({
                    "safe_name": p.name.removesuffix(".SAFE"),
                    "image_time": t
                })

    df = pd.DataFrame(rows)
    if save_csv and not df.empty:
        csv_path = site / f"{site.name}_image_list.csv"
        df[["safe_name", "image_time"]].to_csv(csv_path, index=False)  # ← fixed
        print(f"Saved {len(df)} image names to {csv_path}")
    return df

In [ ]:
# CORSICA
site_folder_corsica = r"M:\SDB_Compare\Corsica"
image_times_corsica = safe_folder_to_df(site_folder_corsica)
print(image_times_corsica[:5])

shapefile_path_corsica = "M:\AOI_new\Corsica_AOI.shp"
gdf_corsica = gpd.read_file(shapefile_path_corsica)
total_bounds_corsica = gdf_corsica.total_bounds
print(f"Total bounds: {total_bounds_corsica}")

crs_corsica = "EPSG:32632"

In [ ]:
# KEY WEST
site_folder_keywest = r"M:\SDB_Compare\Key West"
image_times_keywest = safe_folder_to_df(site_folder_keywest)
print(image_times_keywest[:5])

shapefile_path_keywest = "M:\AOI_new\Key_West_AOI_v2.shp"
gdf_keywest = gpd.read_file(shapefile_path_keywest)
total_bounds_keywest = gdf_keywest.total_bounds
print(f"Total bounds: {total_bounds_keywest}")

crs_keywest = "EPSG:32617"

In [ ]:
# CAPE COD
site_folder_capecod = r"M:\SDB_Compare\Cape Cod"
image_times_capecod = safe_folder_to_df(site_folder_capecod)
print(image_times_capecod[:5])

shapefile_path_capecod = "M:\AOI_new\Cape_Cod_AOI_v2.shp"
gdf_capecod = gpd.read_file(shapefile_path_capecod)
total_bounds_capecod = gdf_capecod.total_bounds
print(f"Total bounds: {total_bounds_capecod}")

crs_capecod = "EPSG:32619"

In [ ]:
# UNGAVA
site_folder_ungava = r"M:\SDB_Compare\Ungava Bay"
image_times_ungava = safe_folder_to_df(site_folder_ungava)
print(image_times_ungava[:5])

shapefile_path_ungava = r"M:\AOI_new\Ungava_Bay_AOI.shp" #r"M:\AOI_new\Ungava_Bay_AOI_prj.shp" 
gdf_ungava = gpd.read_file(shapefile_path_ungava)
total_bounds_ungava = gdf_ungava.total_bounds
print(f"Total bounds: {total_bounds_ungava}")

crs_ungava = "EPSG:4326"

In [ ]:
# -------------------------------------------- DEFINE THE MODEL USED HERE!! -----------------------------
# ----------------------------------------------------"EOT20"--------------------------------------------


def append_tide_heights(csv_path, lon, lat, model="EOT20", directory="tide_models"):
    import time
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)
    df["image_time"] = pd.to_datetime(df["image_time"], utc=True)
    print(f"Tide modelling location: lon={lon:.4f}, lat={lat:.4f}")
    t_start = time.time()
    tides_df = model_tides(
        x=lon,
        y=lat,
        time=df["image_time"],
        model=model,
        directory=directory,
        crs="EPSG:4326",
    ).reset_index()
    df["tide_height_m"] = tides_df["tide_height"].values
    df["tide_model"] = model
    out_path = csv_path.with_stem(f"{csv_path.stem}_{model.replace('-', '_')}")
    df.to_csv(out_path, index=False)
    elapsed = time.time() - t_start
    txt_out = out_path.with_suffix(".txt")
    with open(txt_out, "w") as f:
        f.write(f"output: {out_path.name}\n")
        f.write(f"n_rows: {len(df)}\n")
        f.write(f"model: {model}\n")
        f.write(f"lon: {lon:.4f}, lat: {lat:.4f}\n")
        f.write(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)\n")
    print(f"Saved {len(df)} rows with tide heights ({model}) to {out_path}")
    print(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)")
    return df

In [ ]:

def set_site_vars(sitename):
    global site_folder, bounds, site_crs
    g = globals()
    try:
        site_folder = g[f"site_folder_{sitename}"]
        bounds      = g[f"total_bounds_{sitename}"]
        site_crs    = g[f"crs_{sitename}"]

    except KeyError as e:
        raise KeyError(f"Missing variable for site '{sitename}': {e}")
        
def bounds_centroid(bounds):
    minx, miny, maxx, maxy = bounds
    lon = (minx + maxx) / 2
    lat = (miny + maxy) / 2
    return lon, lat

In [ ]:
# -- Site Names --
# corsica
# keywest
# capecod
# ungava

In [ ]:
# --- UNGAVA ---
set_site_vars("ungava")
lon, lat = bounds_centroid(bounds)
csv_path = Path(site_folder) / f"{Path(site_folder).name}_image_list.csv"

df = append_tide_heights(csv_path, lon=lon, lat=lat)
print(df)

In [ ]:
# --- CORSICA ---
set_site_vars("corsica")
lon, lat = bounds_centroid(bounds)
csv_path = Path(site_folder) / f"{Path(site_folder).name}_image_list.csv"

df = append_tide_heights(csv_path, lon=lon, lat=lat)
print(df)

In [ ]:
# --- CAPE COD ---
set_site_vars("capecod")
lon, lat = bounds_centroid(bounds)
csv_path = Path(site_folder) / f"{Path(site_folder).name}_image_list.csv"

df = append_tide_heights(csv_path, lon=lon, lat=lat)
print(df)

In [ ]:
# --- KEY WEST ---
set_site_vars("keywest")
lon, lat = bounds_centroid(bounds)
csv_path = Path(site_folder) / f"{Path(site_folder).name}_image_list.csv"

df = append_tide_heights(csv_path, lon=lon, lat=lat)
print(df)

In [ ]:
# ------------------- Minimum Tide ------------------- 
site_df = df
print(site_df.loc[site_df["tide_height_m"].idxmin(), "safe_name"])

In [ ]:
def get_lowest_tide_image(sitename):
    csv_path = Path(globals()[f"site_folder_{sitename}"]) 
    csv_path = csv_path / f"{csv_path.name}_image_list.csv"
    
    df = pd.read_csv(csv_path)
    idx = df["tide_height_m"].idxmin()
    row = df.loc[idx]
    
    print(f"Site: {sitename}")
    print(f"Image: {row['safe_name']}")
    print(f"Time:  {row['image_time']}")
    print(f"Tide:  {row['tide_height_m']:.4f} m")
    
    return row["safe_name"]

In [ ]:
# --- Call --- UNGAVA
get_lowest_tide_image("ungava")

In [ ]:
# --- Call --- CORSICA
get_lowest_tide_image("corsica")

In [ ]:
# --- Call --- CAPE COD
get_lowest_tide_image("capecod")

In [ ]:
# --- Call --- KEY WEST
get_lowest_tide_image("keywest")

In [ ]:
def tide_stats(csv_path):
    df = pd.read_csv(csv_path)

    tide_min   = df["tide_height_m"].min()
    tide_max   = df["tide_height_m"].max()
    tide_range = tide_max - tide_min

    print(f"Min:   {tide_min:.4f} m")
    print(f"Max:   {tide_max:.4f} m")
    print(f"Range: {tide_range:.4f} m")

    # Append percentile rank (0-100)
    df["tide_percentile"] = df["tide_height_m"].rank(pct=True) * 100

    df.to_csv(csv_path, index=False)
    print(f"\nSaved percentile ranks to {csv_path}")
    return df

In [ ]:
csv_path = Path(site_folder) / f"{Path(site_folder).name}_image_list_EOT20.csv"
df = tide_stats(csv_path)
print(df[["safe_name", "tide_height_m", "tide_percentile"]])

In [ ]:
csv_path = Path(site_folder_capecod) / f"{Path(site_folder_capecod).name}_image_list_EOT20.csv"
df = tide_stats(csv_path)
print(df[["safe_name", "tide_height_m", "tide_percentile"]])

In [ ]:
csv_path = Path(site_folder_ungava) / f"{Path(site_folder_ungava).name}_image_list_EOT20.csv"
df = tide_stats(csv_path)
print(df[["safe_name", "tide_height_m", "tide_percentile"]])

In [ ]:
csv_path = Path(site_folder_corsica) / f"{Path(site_folder_corsica).name}_image_list_EOT20.csv"
df = tide_stats(csv_path)
print(df[["safe_name", "tide_height_m", "tide_percentile"]])

In [ ]:
# corsica
# capecod
# keywest
# ungava
set_site_vars("corsica")
csv_path2 = Path(site_folder) / f"{Path(site_folder).name}_image_list_EOT20.csv"
make_25p_composite(out_dir=site_folder, csv_path=csv_path2)